# Model B — routed PedViViT (B-soft / B-sparse / FiLM control) on JAAD, round 2

Implements the **routed** half of `idea.md` on top of the PedViViT-style model of notebook A: in the spatial ViT-S encoder, blocks 1-6 are dense and blocks 7-12 carry per-sample (per-clip) **head gates** from one router per layer reading
$z=\mathrm{LN}(\text{frame-averaged CLS after block 6})$.

* **B-soft:** $g=H\,\mathrm{softmax}(s/\tau)$ - nests A at init (zero router).  * **B-sparse:** top-3 of 6 heads, $g=H\,\mathrm{softmax}_{S}(s/\tau)$, noisy selection, first 40% of steps soft (Prop. 1 nesting), balancing + z-loss.
* **FiLM:** unconstrained $\gamma,\beta$ on the concatenated head space (control for Prop. 2: conditioning vs. selection).

**Round 2 protocol (same as notebook A).** Video-grouped 5-fold CV over train+val, fixed-budget training with EMA weights, **the exact fine-tuning recipe selected in notebook A** (`recipe.json`), identical folds / seeds / augmentation for every variant. The official test split is only predicted. H1 compares the fold-ensembled test predictions of each variant with A's by a paired pedestrian-cluster bootstrap.
Round 1's B-vs-A comparison was uninformative because the dense baseline was a one-epoch model; B-sparse additionally collapsed to identical heads for every clip, so its training knobs changed (`routed_vit/recipes.py::SPARSE_EXTRA`: longer soft phase, lower router LR, stronger balancing; these are untested guesses) and the routing diagnostics below show whether that worked.

**Measurements:** H1 performance - H2 routing not collapsed - H3 routes track context given the label (Miller-Madow MI + permutation null) - H4 causal swap tests with a magnitude-matched control - H5 gradient-share specialization.

**Kaggle settings:** GPU T4 x2 - Internet On - Inputs: dataset `christopheronyiuke/jaad-video` and **the output of notebook A** (provides the recipe, A's runs for H1/H5 and the crop cache). Expected run time: about 4-5 h (3 variants x 3 seeds x 5 folds), resumable.

## 1. Code
The `routed_vit` package (model, data, training, analysis) is embedded below with `%%writefile`, so the notebook is self-contained. Only the public JAAD tooling of the base repo (`download_jaad.py`, `build_jaad_samples.py`) is cloned.

In [ ]:
!pip install -q -U timm
!git clone -q --depth 1 https://github.com/Vidit-01/pedestrian_intent_prediction.git /kaggle/working/repo || true
!mkdir -p /kaggle/working/code/routed_vit /kaggle/working/out
!nvidia-smi -L && df -h /kaggle/working /tmp | head -5

In [ ]:
%%writefile /kaggle/working/code/routed_vit/__init__.py
"""Dense vs. routed ViT for pedestrian intent on JAAD (spec: docs/idea.md)."""

In [ ]:
%%writefile /kaggle/working/code/routed_vit/model.py
"""PedViViT-style pedestrian-intent model (VIT-Ped, arXiv 2601.01989) with optional head routing. Spec: idea.md sections 2-4, 6.

PedViViT = ViViT factorised encoder (per-frame spatial ViT -> temporal transformer over per-frame CLS tokens) on the
pedestrian local-context crop, + a small non-visual transformer on delta-encoded bounding boxes (their 'Ours 6': 2 layers,
4 heads), fused by concatenation + FFN into one sigmoid unit. Unspecified in the paper and chosen here: ViT-S/16 spatial encoder
(ImageNet-pretrained), 2-layer temporal transformer, box embedding width 64, FFN width 128. Ego speed is NOT an input
(it is one of the analysed contexts). The routing of idea.md acts only inside the spatial ViT.

variants:  dense  - every head contributes equally
           soft   - B-soft:   g = H * softmax(s / tau)
           sparse - B-sparse: top-k heads, g = H * softmax over the selected logits (noise only picks the set)
           film   - unconstrained FiLM on the concatenated head space (gamma = 1 + W z, beta = W' z), control for Prop. 2
Training-stability additions (round 2): stochastic depth in the spatial ViT, optional freezing of the first blocks, optional
box-only control (no images at all). None of them changes the function at initialisation (Prop. 1 still holds in eval mode).
Routed layers are blocks L0..L-1 (0-based), i.e. 'blocks 7-12' for L0 = 6. One route per clip, from
z = LN(frame-averaged CLS token after block L0).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class Attention(nn.Module):
    """MHA written as a sum of per-head terms: Concat(g_h O_h) W_O = sum_h g_h O_h W_O^(h) (+ bias)."""

    def __init__(self, d, heads):
        super().__init__()
        self.h, self.dk = heads, d // heads
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.record = None   # {'sum': [h,N,dk], 'n': int} -> running mean of per-head outputs (analysis only)
        self.ablate = None   # (head, mean[h,N,dk]) -> mean-ablation of one head (analysis only)

    def forward(self, x, gate=None, film=None):
        B, N, D = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        o = F.scaled_dot_product_attention(q, k, v)  # [B, h, N, dk]
        if self.record is not None:
            self.record['sum'] += o.float().sum(0)
            self.record['n'] += B
        if self.ablate is not None:
            hh, mean = self.ablate
            o = o.clone()
            o[:, hh] = mean.to(o.dtype)
        if gate is not None:
            o = o * gate.to(o.dtype)[:, :, None, None]
        o = o.transpose(1, 2).reshape(B, N, D)
        if film is not None:
            o = o * film[0][:, None, :].to(o.dtype) + film[1][:, None, :].to(o.dtype)
        return self.proj(o)


class Mlp(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(d, hidden), nn.GELU(), nn.Linear(hidden, d)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class DropPath(nn.Module):
    """Stochastic depth: drops the whole residual branch per sample (training only)."""

    def __init__(self, p=0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty(x.shape[0], *([1] * (x.ndim - 1))).bernoulli_(keep)
        return x * mask / keep


class Block(nn.Module):
    def __init__(self, d, heads, mlp_ratio=4, drop_path=0.0):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(d, eps=1e-6), Attention(d, heads)
        self.norm2, self.mlp = nn.LayerNorm(d, eps=1e-6), Mlp(d, d * mlp_ratio)
        self.drop_path = DropPath(drop_path)

    def forward(self, x, gate=None, film=None):
        x = x + self.drop_path(self.attn(self.norm1(x), gate, film))
        return x + self.drop_path(self.mlp(self.norm2(x)))


class PatchEmbed(nn.Module):
    def __init__(self, d, patch=16):
        super().__init__()
        self.proj = nn.Conv2d(3, d, patch, patch)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class Router(nn.Module):
    """s = W z + b (zero-init => uniform gates, Prop. 1). B-sparse also has a noise net (training only)."""

    def __init__(self, d, heads, k, sparse):
        super().__init__()
        self.k = k
        self.w = nn.Linear(d, heads)
        nn.init.zeros_(self.w.weight)
        nn.init.zeros_(self.w.bias)
        self.noise = nn.Linear(d, heads) if sparse else None
        if self.noise is not None:
            nn.init.zeros_(self.noise.weight)
            nn.init.zeros_(self.noise.bias)

    def gate(self, z, tau, sparse_on):
        """-> g [B,H] (sum = H), pi [B,H] (full softmax), mask [B,H] bool or None, s [B,H] logits."""
        with torch.autocast(device_type=z.device.type, enabled=False):
            z = z.float()
            s = self.w(z)
            H = s.shape[-1]
            pi = F.softmax(s / tau, -1)
            if self.noise is None or not sparse_on:
                return H * pi, pi, None, s
            sn = s
            if self.training:
                sn = s + torch.randn_like(s) * F.softplus(self.noise(z))
            idx = sn.topk(self.k, -1).indices
            mask = torch.zeros_like(s, dtype=torch.bool).scatter_(-1, idx, True)
            g = H * F.softmax((s / tau).masked_fill(~mask, float('-inf')), -1)
            return g, pi, mask, s


class RoutedViT(nn.Module):
    def __init__(self, variant='dense', d=384, depth=12, heads=6, L0=6, k=3, T=8, img=224, patch=16,
                 t_layers=2, t_dropout=0.1, use_box=True, d_nv=64, drop_path=0.0, freeze_blocks=0, box_only=False,
                 box_drop=0.0):
        super().__init__()
        assert variant in ('dense', 'soft', 'sparse', 'film')
        assert 0 <= freeze_blocks <= L0, 'only dense blocks (before the routed ones) can be frozen'
        assert not box_only or (variant == 'dense' and use_box), 'box_only is a dense control that needs the box branch'
        self.variant, self.d, self.H, self.L0, self.k, self.T = variant, d, heads, L0, k, T
        self.freeze_blocks, self.box_only, self.box_drop = freeze_blocks, box_only, box_drop
        n_tok = (img // patch) ** 2 + 1
        # shared parameters first, so a fixed seed gives identical shared weights for every variant
        # (stochastic-depth rate grows linearly with depth; frozen blocks are never dropped)
        dp = [0.0 if i < freeze_blocks else drop_path * i / max(1, depth - 1) for i in range(depth)]
        self.patch_embed = PatchEmbed(d, patch)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tok, d))
        self.blocks = nn.ModuleList([Block(d, heads, drop_path=dp[i]) for i in range(depth)])
        self.norm = nn.LayerNorm(d, eps=1e-6)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.t_cls = nn.Parameter(torch.zeros(1, 1, d))
        self.t_pos = nn.Parameter(torch.zeros(1, T + 1, d))
        nn.init.trunc_normal_(self.t_pos, std=0.02)
        nn.init.trunc_normal_(self.t_cls, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, dropout=t_dropout, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, t_layers, enable_nested_tensor=False)
        self.use_box = use_box
        if use_box:   # non-visual branch: delta-encoded boxes -> 2-layer / 4-head transformer -> mean pool
            self.nv_in = nn.Sequential(nn.Linear(4, d_nv), nn.LayerNorm(d_nv))
            self.nv_pos = nn.Parameter(torch.zeros(1, T, d_nv))
            nn.init.trunc_normal_(self.nv_pos, std=0.02)
            nv_layer = nn.TransformerEncoderLayer(d_nv, 4, dim_feedforward=2 * d_nv, dropout=t_dropout, activation='gelu',
                                                  batch_first=True, norm_first=True)
            self.nv = nn.TransformerEncoder(nv_layer, 2, enable_nested_tensor=False)
        d_f = (0 if box_only else d) + (d_nv if use_box else 0)
        self.fuse = nn.Sequential(nn.LayerNorm(d_f), nn.Linear(d_f, 128), nn.GELU(), nn.Dropout(t_dropout), nn.Linear(128, 1))
        # routing parameters (variant specific), created last
        n_r = depth - L0
        if variant != 'dense':
            self.z_norm = nn.LayerNorm(d)
        if variant in ('soft', 'sparse'):
            self.routers = nn.ModuleList([Router(d, heads, k, variant == 'sparse') for _ in range(n_r)])
        if variant == 'film':
            self.film = nn.ModuleList([nn.Linear(d, 2 * d) for _ in range(n_r)])
            for m in self.film:
                nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)

        if freeze_blocks:
            for m in (self.patch_embed, self.blocks[:freeze_blocks]):
                for p in m.parameters():
                    p.requires_grad_(False)
            self.cls_token.requires_grad_(False)
            self.pos_embed.requires_grad_(False)

    def load_pretrained(self, name='vit_small_patch16_224.augreg_in21k_ft_in1k'):
        import timm
        sd = timm.create_model(name, pretrained=True).state_dict()
        sd = {k: v for k, v in sd.items() if not k.startswith(('head', 'fc_norm'))}
        missing, unexpected = self.load_state_dict(sd, strict=False)
        bad = [m for m in missing if m.split('.')[0] in ('patch_embed', 'cls_token', 'pos_embed', 'blocks', 'norm')]
        assert not bad and not unexpected, f'pretrained load mismatch: missing={bad} unexpected={unexpected}'

    @property
    def n_routed(self):
        return len(self.blocks) - self.L0

    def embed_to_L0(self, x):
        """x [B,T,3,H,W] -> tokens after the last dense block, [B*T, N, d]."""
        nf = self.freeze_blocks
        with torch.set_grad_enabled(torch.is_grad_enabled() and nf == 0):   # frozen prefix: no activations kept for backward
            tok = self.patch_embed(x.flatten(0, 1))
            tok = torch.cat([self.cls_token.expand(tok.shape[0], -1, -1), tok], 1) + self.pos_embed
            for blk in self.blocks[:nf]:
                tok = blk(tok)
        for blk in self.blocks[nf:self.L0]:
            tok = blk(tok)
        return tok

    def routed_forward(self, tok, B, T, tau=1.0, sparse_on=True, gates=None, box=None):
        """tok: [B*T,N,d] after block L0. gates: optional list of [B,H] overriding the router (swap tests)."""
        zero = tok.new_zeros(()).float()
        info = {'gates': [], 'pi': [], 'mask': [], 'bal': zero, 'zloss': zero}
        z = None
        if self.variant != 'dense':
            with torch.autocast(device_type=tok.device.type, enabled=False):
                z = self.z_norm(tok[:, 0].float().view(B, T, -1).mean(1))
            info['z'] = z
        for li in range(self.L0, len(self.blocks)):
            blk, r = self.blocks[li], li - self.L0
            gate = film = None
            if self.variant == 'film':
                gb = self.film[r](z)
                film = ((1 + gb[:, :self.d]).repeat_interleave(T, 0), gb[:, self.d:].repeat_interleave(T, 0))
            elif self.variant in ('soft', 'sparse'):
                if gates is not None:
                    g = gates[r]
                else:
                    g, pi, mask, s = self.routers[r].gate(z, tau, sparse_on)
                    info['pi'].append(pi)
                    info['mask'].append(mask)
                    info['zloss'] = info['zloss'] + (torch.logsumexp(s, -1) ** 2).mean()
                    if mask is not None:   # B-sparse: H * sum_h f_h P_h
                        info['bal'] = info['bal'] + self.H * ((mask.float().mean(0) / self.k) * pi.mean(0)).sum()
                    else:                  # B-soft: CV^2 of the per-head load
                        load = g.sum(0)
                        info['bal'] = info['bal'] + load.var(unbiased=False) / (load.mean() ** 2 + 1e-8)
                info['gates'].append(g)
                gate = g.repeat_interleave(T, 0)
            tok = blk(tok, gate, film)
        info['xL'] = tok[:, 0].float().view(B, T, -1).mean(1)   # residual-stream CLS at the last block (pre final LN)
        cls = self.norm(tok)[:, 0].view(B, T, -1)
        h = torch.cat([self.t_cls.expand(B, -1, -1).to(cls.dtype), cls], 1) + self.t_pos.to(cls.dtype)
        f = self.temporal(h)[:, 0]
        if self.use_box:
            fb = self.nv(self.nv_in(box.to(f.dtype)) + self.nv_pos.to(f.dtype)).mean(1)
            if self.training and self.box_drop > 0:   # modality dropout: stops the visual branch from being ignored
                fb = fb * (torch.rand(fb.shape[0], 1, device=fb.device) >= self.box_drop).to(fb.dtype)
            f = torch.cat([f, fb], 1)
        return self.fuse(f).squeeze(-1), info

    def box_only_forward(self, box):
        """Control: the non-visual branch alone (no pixels). Tells how much of the signal is bounding-box dynamics."""
        z = box.new_zeros(()).float()
        h = self.nv_in(box)
        f = self.nv(h + self.nv_pos.to(h.dtype)).mean(1)
        return self.fuse(f).squeeze(-1), {'gates': [], 'pi': [], 'mask': [], 'bal': z, 'zloss': z}

    def forward(self, x, box=None, tau=1.0, sparse_on=True, gates=None):
        if self.box_only:
            return self.box_only_forward(box)
        B, T = x.shape[:2]
        return self.routed_forward(self.embed_to_L0(x), B, T, tau, sparse_on, gates, box)


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    rout = sum(p.numel() for n, p in model.named_parameters() if n.startswith(('routers', 'film', 'z_norm')))
    return tot, rout

In [ ]:
%%writefile /kaggle/working/code/routed_vit/metrics.py
"""Metrics mirroring the repo's evaluate.py (kept separate: `evaluate` collides with the HF package on Kaggle),
plus cluster (per-pedestrian) bootstrap CIs and a paired bootstrap for AUC differences."""
import numpy as np
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss, f1_score,
                             precision_score, recall_score, roc_auc_score)


def select_threshold(y, p):
    """Threshold maximising balanced accuracy; call on VALIDATION predictions only."""
    cands = np.unique(np.concatenate([p, [0.5]]))
    return float(cands[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in cands]))])


def _thr(y, p, thr, pre):
    yh = (p >= thr).astype(int)
    return {pre + 'acc': accuracy_score(y, yh), pre + 'balanced_acc': balanced_accuracy_score(y, yh),
            pre + 'f1': f1_score(y, yh, zero_division=0), pre + 'precision': precision_score(y, yh, zero_division=0),
            pre + 'recall': recall_score(y, yh, zero_division=0)}


def compute_metrics(y, p, thr=0.5):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=np.float64)
    two = len(np.unique(y)) == 2
    m = {'n': len(y), 'pos': int(y.sum()), 'auc': roc_auc_score(y, p) if two else float('nan'),
         'bench_auc': roc_auc_score(y, np.round(p)) if two else float('nan'), 'brier': brier_score_loss(y, p),
         'threshold': thr}
    m.update(_thr(y, p, 0.5, 'bench_'))
    m.update(_thr(y, p, thr, 'thr_'))
    return {k: float(v) for k, v in m.items()}


def _clusters(groups):
    _, inv = np.unique(groups, return_inverse=True)
    return [np.where(inv == g)[0] for g in range(inv.max() + 1)]


def bootstrap_metrics(y, p, groups, thr, n_boot=1000, seed=0, keys=('auc', 'bench_acc', 'bench_f1', 'thr_f1')):
    """95% percentile CIs, resampling whole pedestrians (windows of one pedestrian are highly correlated)."""
    rng = np.random.default_rng(seed)
    cl = _clusters(groups)
    pt = compute_metrics(y, p, thr)
    boots = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = np.concatenate([cl[i] for i in rng.integers(0, len(cl), len(cl))])
        if len(np.unique(y[idx])) < 2:
            continue
        m = compute_metrics(y[idx], p[idx], thr)
        for k in keys:
            boots[k].append(m[k])
    return {k: (pt[k], float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in boots.items()}


def paired_auc_diff(y, preds_a, preds_b, groups, n_boot=1000, seed=0):
    """AUC_B - AUC_A averaged over seeds (preds_*: list of [n] arrays, one per seed), same pedestrian resamples."""
    rng = np.random.default_rng(seed)
    cl = _clusters(groups)
    d0 = np.mean([roc_auc_score(y, b) for b in preds_b]) - np.mean([roc_auc_score(y, a) for a in preds_a])
    ds = []
    for _ in range(n_boot):
        idx = np.concatenate([cl[i] for i in rng.integers(0, len(cl), len(cl))])
        if len(np.unique(y[idx])) < 2:
            continue
        ds.append(np.mean([roc_auc_score(y[idx], b[idx]) for b in preds_b]) -
                  np.mean([roc_auc_score(y[idx], a[idx]) for a in preds_a]))
    return float(d0), float(np.percentile(ds, 2.5)), float(np.percentile(ds, 97.5))

In [ ]:
%%writefile /kaggle/working/code/routed_vit/cv.py
"""Cross-validation protocol shared by the ViT study and Pedestrian Graph+.

Why: the official JAAD_beh validation split has 22 pedestrians. Selecting epochs / hyper-parameters on it is close to random
(val AUC 0.8 vs test AUC 0.6 in the first round of experiments). Instead, train+val are pooled (216 pedestrians) and split into
K folds grouped by *video* (pedestrians of one video share the scene, so they must not straddle train and validation).
The official test split is never used for any selection; it is only predicted by the fold models and averaged.
"""
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold


def pool_trainval(data):
    """Concatenate the 'train' and 'val' records of a samples pickle into one record (rows: train first, then val)."""
    a, b = data['train'], data['val']
    out = {}
    for k in a:
        out[k] = a[k] + b[k] if isinstance(a[k], list) else np.concatenate([a[k], b[k]])
    return out


def ped_groups(rec):
    return np.array([f'{v}/{p}' for v, p in zip(rec['vid'], rec['pid'])])


def make_folds(rec, n_folds=5, seed=0):
    """List of (train_rows, val_rows) over the pooled record. Grouped by video, stratified by label, deterministic in `seed`.
    The fold assignment is independent of the training seed and of the model variant, so every model sees identical folds."""
    vid = np.array(rec['vid'])
    y = np.asarray(rec['label'])
    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(np.sort(tr), np.sort(va)) for tr, va in skf.split(np.zeros(len(y)), y, vid)]


def describe_folds(rec, folds):
    g, y = ped_groups(rec), np.asarray(rec['label'])
    rows = []
    for i, (tr, va) in enumerate(folds):
        rows.append({'fold': i, 'train_windows': len(tr), 'val_windows': len(va), 'train_peds': len(set(g[tr])),
                     'val_peds': len(set(g[va])), 'val_pos_rate': round(float(y[va].mean()), 3)})
    return rows

In [ ]:
%%writefile /kaggle/working/code/routed_vit/recipes.py
"""Fine-tuning recipes for the dense-ViT recipe search (notebook A). Round 1 (lr 5e-5, no layer decay, early stopping on 22
validation pedestrians) peaked at epoch 1 and then memorised the training set. These four trade off how far the pretrained
ViT may move (LR, layer-wise decay, frozen prefix) against regularisation strength. Notebook A runs all of them with 5-fold
video-grouped CV (seed 42), picks the one with the best pooled out-of-fold AUC, and notebook B re-uses that exact recipe.
Every other training argument is the default of `routed_vit.train`.
"""
RECIPES = {
    'r1_lr2e-5': ['--epochs', 10, '--lr', 2e-5, '--llrd', 0.8],
    'r2_lr1e-5_long': ['--epochs', 14, '--lr', 1e-5, '--llrd', 0.85],
    'r3_frozen6': ['--epochs', 10, '--lr', 5e-5, '--llrd', 0.9, '--freeze_blocks', 6],
    'r4_strong_reg': ['--epochs', 8, '--lr', 2e-5, '--llrd', 0.75, '--drop_path', 0.2, '--label_smooth', 0.1,
                      '--box_drop', 0.3, '--box_noise', 0.04],
}
DEFAULT_RECIPE = 'r1_lr2e-5'

# B-sparse collapsed in round 1 (every clip picked the same 3 heads). Untested guesses to break that: longer soft phase,
# lower router LR, stronger balancing loss. Notebook B prints the routing diagnostics so they can be judged.
SPARSE_EXTRA = ['--sparse_start', 0.4, '--router_lr_mult', 5, '--lam_bal', 0.05]

In [ ]:
%%writefile /kaggle/working/code/routed_vit/data.py
"""JAAD data for the routed-ViT study: PCPA-protocol samples (+ annotated context), pedestrian-crop cache, Dataset.

Samples come from the repo's build_jaad_samples.build (obs 16 frames, TTE 30-60, stride 3, official video split).
Each clip uses T=8 of the 16 observed frames (odd indices 1,3,...,15: last observed frame included).
Context c (analysis only, never a model input) is read from the JAAD annotations; see CTX_NAMES.
"""
import json
import os
import pickle
import sys
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

FRAME_IDX = tuple(range(1, 16, 2))
CTX_NAMES = ['signalized', 'intersection', 'designated', 'group_size', 'num_lanes', 'ego_act', 'traffic_light',
             'ped_crossing']
CTX_LEVELS = {'signalized': ['n/a', 'NS', 'S'], 'intersection': ['no', 'yes'], 'designated': ['ND', 'D'],
              'group_size': ['1', '2', '3+'], 'num_lanes': ['1', '2', '3', '4+'],
              'ego_act': ['stopped', 'slow', 'fast', 'decel', 'accel'], 'traffic_light': ['n/a', 'red', 'green'],
              'ped_crossing': ['no', 'yes']}
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)


# ----------------------------------------------------------------------------- samples + context
def build_samples(repo_root, jaad_root, out_pkl, sample_type='beh'):
    """PCPA samples via the repo's builder, plus per-window context array rec['ctx'] [n, len(CTX_NAMES)] (int)."""
    sys.path.insert(0, repo_root)
    from build_jaad_samples import build
    data = build(jaad_root, sample_type)
    from jaad_data import JAAD   # build() put jaad_root on sys.path
    db = JAAD(data_path=jaad_root).generate_database()
    for split in ['train', 'val', 'test']:
        rec = data[split]
        ctx = np.zeros((len(rec['label']), len(CTX_NAMES)), dtype=np.int64)
        for i in range(len(rec['label'])):
            vid, pid = rec['vid'][i], rec['pid'][i]
            a = db[vid]['ped_annotations'][pid].get('attributes', {})
            t = db[vid]['traffic_annotations'].get(int(rec['frames'][i][-1]), {})
            ctx[i] = [int(a.get('signalized', 0)), int(a.get('intersection', 0)), int(a.get('designated', 0)),
                      min(max(int(a.get('group_size', 1)), 1), 3) - 1, min(max(int(a.get('num_lanes', 2)), 1), 4) - 1,
                      int(rec['vehicle_act'][i][-1]), int(t.get('traffic_light', 0)), int(t.get('ped_crossing', 0))]
        rec['ctx'] = ctx
    data['meta']['ctx_names'] = CTX_NAMES
    data['meta']['frame_idx'] = list(FRAME_IDX)
    os.makedirs(os.path.dirname(out_pkl), exist_ok=True)
    with open(out_pkl, 'wb') as f:
        pickle.dump(data, f)
    return data


def load_samples(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def groups_of(rec):
    return np.array([f'{v}/{p}' for v, p in zip(rec['vid'], rec['pid'])])


# ----------------------------------------------------------------------------- crop cache
def square_crop(img, box, scale, size):
    import cv2
    x1, y1, x2, y2 = [float(b) for b in box]
    s = max(max(x2 - x1, y2 - y1) * scale, 16.0)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    l, t = int(round(cx - s / 2)), int(round(cy - s / 2))
    side = int(round(s))
    H, W = img.shape[:2]
    patch = np.zeros((side, side, 3), np.uint8)
    xa, ya, xb, yb = max(l, 0), max(t, 0), min(l + side, W), min(t + side, H)
    if xb > xa and yb > ya:
        patch[ya - t:yb - t, xa - l:xb - l] = img[ya:yb, xa:xb]
    return cv2.resize(patch, (size, size), interpolation=cv2.INTER_AREA if side > size else cv2.INTER_LINEAR)


def _crop_worker(job):
    import cv2
    clips_dir, out_dir, vid, items, size, scale = job
    npy, js = os.path.join(out_dir, vid + '.npy'), os.path.join(out_dir, vid + '.json')
    if os.path.exists(npy) and os.path.exists(js):
        return vid, 'skip'
    by_frame = {}
    for pid, fr, box in items:
        by_frame.setdefault(fr, []).append((pid, box))
    last = max(by_frame)
    cap = cv2.VideoCapture(os.path.join(clips_dir, vid + '.mp4'))
    keys, crops = [], []
    for i in range(last + 1):
        if not cap.grab():
            raise RuntimeError(f'{vid}: could not read frame {i} (need up to {last})')
        if i in by_frame:
            ok, bgr = cap.retrieve()
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            for pid, box in by_frame[i]:
                keys.append([pid, i])
                crops.append(square_crop(rgb, box, scale, size))
    cap.release()
    np.save(npy + '.tmp.npy', np.stack(crops))
    os.replace(npy + '.tmp.npy', npy)
    with open(js, 'w') as f:
        json.dump(keys, f)
    return vid, len(keys)


def build_crop_cache(data, clips_dir, out_dir, size=224, scale=1.5, frame_idx=FRAME_IDX, workers=4):
    """One uint8 [n,size,size,3] npy + key json per video, containing exactly the (pid, frame) crops the clips use."""
    os.makedirs(out_dir, exist_ok=True)
    need = {}
    for split in ['train', 'val', 'test']:
        rec = data[split]
        for i in range(len(rec['label'])):
            d = need.setdefault(rec['vid'][i], {})
            for j in frame_idx:
                d[(rec['pid'][i], int(rec['frames'][i][j]))] = rec['bbox'][i][j]
    jobs = [(clips_dir, out_dir, v, [(p, f, b) for (p, f), b in d.items()], size, scale) for v, d in need.items()]
    done = 0
    with Pool(workers) as pool:
        for vid, n in pool.imap_unordered(_crop_worker, jobs):
            done += 1
            if done % 20 == 0 or done == len(jobs):
                print(f'  crops {done}/{len(jobs)} videos ({vid}: {n})', flush=True)
    return out_dir


# ----------------------------------------------------------------------------- dataset
class CropDataset(Dataset):
    """Clips of a record `rec`. `indices` restricts it to some rows (cross-validation folds of the pooled train+val record);
    the returned 'idx' is always the row in `rec`, so predictions can be mapped back. images=False skips the pixels (box-only control)."""

    def __init__(self, rec, root, frame_idx=FRAME_IDX, indices=None, images=True):
        self.rec, self.root, self.fi, self.images = rec, root, list(frame_idx), images
        self.rows = np.arange(len(rec['label'])) if indices is None else np.asarray(indices)
        self.arrays, self.index = {}, {}

    def __len__(self):
        return len(self.rows)

    def _load(self, vid):
        if vid not in self.arrays:
            self.arrays[vid] = np.load(os.path.join(self.root, vid + '.npy'), mmap_mode='r')
            with open(os.path.join(self.root, vid + '.json')) as f:
                self.index[vid] = {(p, int(fr)): r for r, (p, fr) in enumerate(json.load(f))}
        return self.arrays[vid], self.index[vid]

    def __getitem__(self, k):
        i = int(self.rows[k])
        vid, pid = self.rec['vid'][i], self.rec['pid'][i]
        if self.images:
            arr, idx = self._load(vid)
            rows = [idx[(pid, int(self.rec['frames'][i][j]))] for j in self.fi]
            x = torch.from_numpy(np.ascontiguousarray(arr[rows])).permute(0, 3, 1, 2)   # [T,3,H,W] uint8
        else:
            x = torch.zeros(len(self.fi), 3, 1, 1, dtype=torch.uint8)
        wh = np.asarray(self.rec['img_wh'][i], dtype=np.float32)
        bb = self.rec['bbox'][i][self.fi] / np.concatenate([wh, wh])
        box = torch.from_numpy(((bb - bb[0]) * 5.0).astype(np.float32))   # delta-encoded vs the first frame (PedViViT)
        return {'x': x, 'box': box, 'y': torch.tensor(float(self.rec['label'][i])), 'idx': i}


def _geo_aug(x, strength, gen=None):
    """Same random zoom (0.9-1.1) / shift (+-6%) / rotation (+-5 deg) for all frames of a clip (x: [B,T,3,H,W] float)."""
    B, T, C, H, W = x.shape
    dev = x.device
    u = lambda lo, hi: lo + (hi - lo) * torch.rand(B, device=dev, generator=gen)
    sc = 1.0 + strength * u(-0.1, 0.1)
    ang = strength * u(-5.0, 5.0) * 3.14159265 / 180
    tx, ty = strength * u(-0.12, 0.12), strength * u(-0.12, 0.12)    # normalised coords: 0.12 = 6% of the width
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)   # [B,2,3]
    grid = F.affine_grid(theta, (B, T * C, H, W), align_corners=False)
    out = F.grid_sample(x.reshape(B, T * C, H, W), grid, mode='bilinear', padding_mode='reflection', align_corners=False)
    return out.reshape(B, T, C, H, W)


def prep(x, train=False, gen=None, geo=1.0):
    """uint8 [B,T,3,H,W] (on the model device) -> normalised float.
    Train aug: hflip + brightness/contrast per clip, plus a clip-consistent small affine jitter (strength `geo`, 0 disables)."""
    x = x.float() / 255.0
    if train:
        B = x.shape[0]
        flip = torch.rand(B, device=x.device, generator=gen) < 0.5
        x = torch.where(flip[:, None, None, None, None], x.flip(-1), x)
        b = 0.8 + 0.4 * torch.rand(B, 1, 1, 1, 1, device=x.device, generator=gen)
        c = 0.8 + 0.4 * torch.rand(B, 1, 1, 1, 1, device=x.device, generator=gen)
        m = x.mean(dim=(1, 2, 3, 4), keepdim=True)
        x = ((x - m) * c + m) * b
        if geo > 0:
            x = _geo_aug(x, geo, gen)
        x = x.clamp(0, 1)
    return (x - MEAN.to(x.device)) / STD.to(x.device)


def context_table(data):
    """Tidy table: one row per context variable, one column per split with 'level:count' strings."""
    import pandas as pd
    rows = []
    for k, name in enumerate(CTX_NAMES):
        row = {'context': name}
        for split in ['train', 'val', 'test']:
            counts = np.bincount(data[split]['ctx'][:, k], minlength=len(CTX_LEVELS[name]))
            row[split] = '  '.join(f'{lv}:{c}' for lv, c in zip(CTX_LEVELS[name], counts))
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
%%writefile /kaggle/working/code/routed_vit/train.py
"""Train + evaluate one (variant, seed, fold) job of the PedViViT-style backbone on one GPU.   (idea.md sections 6 and 10)

    CUDA_VISIBLE_DEVICES=0 python -m routed_vit.train --variant soft --seed 42 --fold 0 --samples S.pkl --crops C --out runs/soft_s42_f0

Protocol (round 2). Round 1 selected the best epoch on the 22-pedestrian validation split and early-stopped; every run peaked at
epoch 1 and then memorised the training set (train loss 1e-5), i.e. the "trained" model was a one-epoch checkpoint. Now:
  * train+val are pooled and split into K video-grouped folds (cv.py); `--fold i` trains on K-1 folds and validates on the held-out one;
    `--fold -1` reproduces the official train/val split. The official test split is only predicted, never used for selection.
  * FIXED budget, no early stopping, no best-epoch selection: warm-up -> cosine to ~0 over `--epochs`, and the model that is
    evaluated is the EMA of the weights at the END of the schedule. The same rule applies to every variant.
  * regularisation against memorisation: lower LR with layer-wise decay (`--llrd`), optional frozen prefix (`--freeze_blocks`),
    stochastic depth, label smoothing, EMA, clip-consistent affine jitter, box noise, box-branch modality dropout.
Identical for every variant: pretrained init, folds, augmentation, optimizer, schedule, step count, seeds, selection rule (none).
Writes to --out: config.json, best.pt (final EMA weights, fp16), history.csv, metrics.json, test_outputs.npz, val_outputs.npz
(+ test_curve.npz with --track_test: diagnostic only, never used for selection).
"""
import argparse
import csv
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from .cv import make_folds, pool_trainval
from .data import CropDataset, load_samples, prep
from .metrics import compute_metrics
from .model import RoutedViT, count_params


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def entropy(p, axis=-1):
    return -(p * np.log(p + 1e-12)).sum(axis)


@torch.no_grad()
def predict(model, loader, device, amp=True, tau=1.0):
    """Returns dict of arrays: idx, y, logit, p and, for routed variants, z [n,d], gates/pi [n,Lr,H], mask [n,Lr,H]."""
    model.eval()
    out = {k: [] for k in ['idx', 'y', 'logit', 'z', 'gates', 'pi', 'mask']}
    for b in loader:
        x = None if model.box_only else prep(b['x'].to(device, non_blocking=True))
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp and device.type == 'cuda'):
            logit, info = model(x, box=b['box'].to(device), tau=tau)
        out['idx'].append(b['idx'].numpy())
        out['y'].append(b['y'].numpy())
        out['logit'].append(logit.float().cpu().numpy())
        if info['gates']:
            out['z'].append(info['z'].cpu().numpy().astype(np.float16))
            out['gates'].append(torch.stack(info['gates'], 1).cpu().numpy())
            if info['pi']:
                out['pi'].append(torch.stack(info['pi'], 1).cpu().numpy())
            if info['mask'] and info['mask'][0] is not None:
                out['mask'].append(torch.stack(info['mask'], 1).cpu().numpy())
    res = {k: np.concatenate(v) for k, v in out.items() if v}
    res['p'] = 1 / (1 + np.exp(-res['logit']))
    return res


class EMA:
    """Exponential moving average of the parameters (fp32 shadow). The model has no buffers, so parameters are the whole state."""

    def __init__(self, model, decay):
        self.decay = decay
        self.params = [p for p in model.parameters()]
        self.shadow = [p.detach().clone().float() for p in self.params]
        self.backup = None

    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step))    # warm start: do not average in the random-ish first steps
        torch._foreach_mul_(self.shadow, d)
        torch._foreach_add_(self.shadow, [p.detach().float() for p in self.params], alpha=1 - d)

    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s in zip(self.params, self.shadow):
            p.copy_(s.to(p.dtype))

    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup):
            p.copy_(b)
        self.backup = None


def layer_id(name, depth):
    """Depth index for layer-wise LR decay: embeddings 0, block i -> i+1, final norm depth+1."""
    if name.startswith(('patch_embed', 'cls_token', 'pos_embed')):
        return 0
    if name.startswith('blocks.'):
        return int(name.split('.')[1]) + 1
    return depth + 1


def make_optimizer(model, args):
    depth = len(model.blocks)
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith(('routers', 'film', 'z_norm')):
            g, lr = 'router', args.lr * args.router_lr_mult
        elif n.startswith(('temporal', 't_cls', 't_pos', 'nv', 'fuse')):
            g, lr = 'head', args.head_lr
        else:
            g, lr = 'backbone', args.lr * args.llrd ** (depth + 1 - layer_id(n, depth))
        decay = p.ndim > 1 and not n.endswith(('pos_embed', 't_pos', 'nv_pos', 'cls_token', 't_cls'))
        groups.setdefault((g, round(lr, 12), decay), []).append(p)
    pg = [{'params': ps, 'weight_decay': args.wd if decay else 0.0, 'group': g, 'base_lr': lr, 'lr': lr}
          for (g, lr, decay), ps in groups.items()]
    return torch.optim.AdamW(pg, betas=(0.9, 0.999))


def lr_scale(step, total, warmup):
    w = max(1, int(warmup * total))
    if step < w:
        return (step + 1) / w
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - w) / max(1, total - w)))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--variant', required=True, choices=['dense', 'soft', 'sparse', 'film'])
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--fold', type=int, default=0, help='CV fold held out for validation; -1 = official train/val split')
    ap.add_argument('--n_folds', type=int, default=5)
    ap.add_argument('--fold_seed', type=int, default=0, help='fold assignment seed (same for every variant / training seed)')
    ap.add_argument('--samples', required=True)
    ap.add_argument('--crops', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--epochs', type=int, default=10)
    ap.add_argument('--micro_bs', type=int, default=16, help='clips per forward (x8 frames = images)')
    ap.add_argument('--accum', type=int, default=1)
    ap.add_argument('--lr', type=float, default=2e-5, help='backbone LR at the top block (lower blocks decay by --llrd per block)')
    ap.add_argument('--llrd', type=float, default=0.8, help='layer-wise LR decay of the spatial ViT')
    ap.add_argument('--head_lr', type=float, default=5e-4)
    ap.add_argument('--router_lr_mult', type=float, default=10.0)
    ap.add_argument('--wd', type=float, default=0.05)
    ap.add_argument('--warmup', type=float, default=0.1)
    ap.add_argument('--router_freeze', type=float, default=0.05, help='fraction of steps with the router frozen')
    ap.add_argument('--L0', type=int, default=6)
    ap.add_argument('--k', type=int, default=3)
    ap.add_argument('--lam_bal', type=float, default=1e-2)
    ap.add_argument('--lam_z', type=float, default=1e-3)
    ap.add_argument('--tau0', type=float, default=2.0)
    ap.add_argument('--tau_frac', type=float, default=0.2, help='tau anneals tau0 -> 1 over this fraction of steps')
    ap.add_argument('--sparse_start', type=float, default=0.3,
                    help='B-sparse: first this fraction of steps runs soft (Prop. 1 nesting), then switches to top-k')
    ap.add_argument('--freeze_blocks', type=int, default=0, help='freeze patch embedding + first N ViT blocks (N <= L0)')
    ap.add_argument('--drop_path', type=float, default=0.1, help='stochastic depth, linear to this value at the last block')
    ap.add_argument('--label_smooth', type=float, default=0.05)
    ap.add_argument('--ema', type=float, default=0.995, help='EMA decay of the weights (0 disables); the EMA model is evaluated')
    ap.add_argument('--geo_aug', type=float, default=1.0, help='strength of the clip-consistent affine jitter (0 disables)')
    ap.add_argument('--box_noise', type=float, default=0.02, help='gaussian noise on the delta-encoded boxes (train only)')
    ap.add_argument('--box_drop', type=float, default=0.15, help='probability of zeroing the box-branch feature (train only)')
    ap.add_argument('--workers', type=int, default=4)
    ap.add_argument('--no_pretrained', action='store_true')
    ap.add_argument('--no_amp', action='store_true')
    ap.add_argument('--no_box', action='store_true', help='drop the non-visual (bbox) branch')
    ap.add_argument('--box_only', action='store_true', help='control: bbox branch only, no pixels (dense variant)')
    ap.add_argument('--track_test', action='store_true', help='diagnostic: also store test predictions after every epoch')
    ap.add_argument('--max_train', type=int, default=0, help='debug: use only this many train clips')
    args = ap.parse_args(argv)

    if os.path.exists(os.path.join(args.out, 'metrics.json')):
        print(f'[skip] {args.out} already finished')
        return
    os.makedirs(args.out, exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = device.type == 'cuda' and not args.no_amp
    set_seed(args.seed)

    data = load_samples(args.samples)
    pool = pool_trainval(data)
    if args.fold < 0:
        n_tr = len(data['train']['label'])
        tr_rows, va_rows = np.arange(n_tr), np.arange(n_tr, len(pool['label']))
    else:
        tr_rows, va_rows = make_folds(pool, args.n_folds, args.fold_seed)[args.fold]
    if args.max_train:
        tr_rows = tr_rows[:args.max_train]
    img = not args.box_only
    train_ds = CropDataset(pool, args.crops, indices=tr_rows, images=img)
    val_ds = CropDataset(pool, args.crops, indices=va_rows, images=img)
    test_ds = CropDataset(data['test'], args.crops, images=img)
    g = torch.Generator()
    g.manual_seed(args.seed * 100 + max(args.fold, 0))
    tl = DataLoader(train_ds, args.micro_bs, shuffle=True, drop_last=True, num_workers=args.workers, generator=g,
                    pin_memory=True, persistent_workers=args.workers > 0)
    ev = {'val': DataLoader(val_ds, 32, shuffle=False, num_workers=args.workers, pin_memory=True),
          'test': DataLoader(test_ds, 32, shuffle=False, num_workers=args.workers, pin_memory=True)}

    y_tr = pool['label'][tr_rows]
    w = {1: len(y_tr) / (2 * max(1, y_tr.sum())), 0: len(y_tr) / (2 * max(1, (1 - y_tr).sum()))}   # inverse frequency

    set_seed(args.seed)   # identical shared-weight init for every variant
    model = RoutedViT(args.variant, L0=args.L0, k=args.k, use_box=not args.no_box, drop_path=args.drop_path,
                      freeze_blocks=args.freeze_blocks, box_only=args.box_only, box_drop=args.box_drop)
    if not args.no_pretrained and not args.box_only:
        model.load_pretrained()
    model.to(device)
    n_params, n_router = count_params(model)
    opt = make_optimizer(model, args)
    ema = EMA(model, args.ema) if args.ema > 0 else None
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    steps_per_epoch = len(tl) // args.accum
    total = args.epochs * steps_per_epoch
    routed = args.variant in ('soft', 'sparse')
    print(f'{args.variant} seed={args.seed} fold={args.fold} train={len(tr_rows)} val={len(va_rows)} params={n_params / 1e6:.2f}M '
          f'(router/film {n_router / 1e3:.1f}k) steps={total} clips/step={args.micro_bs * args.accum}', flush=True)

    def evaluate(split):
        if ema:
            ema.apply()
        try:
            return predict(model, ev[split], device, amp)
        finally:
            if ema:
                ema.restore()

    hist, step, test_curve = [], 0, []
    t0 = time.time()
    for ep in range(args.epochs):
        model.train()
        run = {'loss': 0.0, 'task': 0.0, 'n': 0}
        it = iter(tl)
        for _ in range(steps_per_epoch):
            frac = step / total
            sc = lr_scale(step, total, args.warmup)
            for pg in opt.param_groups:
                frozen = pg['group'] == 'router' and frac < args.router_freeze
                pg['lr'] = 0.0 if frozen else pg['base_lr'] * sc
            tau = max(1.0, args.tau0 - (args.tau0 - 1.0) * min(1.0, frac / max(args.tau_frac, 1e-8)))
            sparse_on = frac >= args.sparse_start
            opt.zero_grad(set_to_none=True)
            for _ in range(args.accum):
                b = next(it)
                x = None if args.box_only else prep(b['x'].to(device, non_blocking=True), train=True, geo=args.geo_aug)
                box = b['box'].to(device)
                if args.box_noise > 0:
                    box = box + args.box_noise * torch.randn_like(box)
                y = b['y'].to(device)
                with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
                    logit, info = model(x, box=box, tau=tau, sparse_on=sparse_on)
                sw = torch.where(y > 0.5, torch.tensor(w[1], device=device), torch.tensor(w[0], device=device))
                ys = y * (1 - args.label_smooth) + 0.5 * args.label_smooth
                task = (F.binary_cross_entropy_with_logits(logit.float(), ys, reduction='none') * sw).mean()
                loss = task + (args.lam_bal * info['bal'] + args.lam_z * info['zloss'] if routed else 0.0)
                scaler.scale(loss / args.accum).backward()
                run['loss'] += float(loss.detach()) * len(y)
                run['task'] += float(task.detach()) * len(y)
                run['n'] += len(y)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            step += 1
            if ema:
                ema.update(step)
        vo = evaluate('val')
        m = compute_metrics(vo['y'], vo['p'])
        S = float('nan')
        if 'pi' in vo:
            pi = vo['pi']
            S = float(np.mean(entropy(pi.mean(0)) - entropy(pi).mean(0)))
        row = {'epoch': ep + 1, 'train_loss': run['loss'] / run['n'], 'train_task': run['task'] / run['n'],
               'val_auc': m['auc'], 'val_bench_auc': m['bench_auc'], 'val_f1': m['bench_f1'], 'route_S': S,
               'time_min': (time.time() - t0) / 60}
        if args.track_test:
            to_ep = evaluate('test')
            test_curve.append(to_ep['p'])
            row['test_auc_diag'] = compute_metrics(to_ep['y'], to_ep['p'])['auc']
        hist.append(row)
        print(' '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in row.items()), flush=True)

    # final model = EMA weights at the end of the fixed schedule (no checkpoint selection)
    if ema:
        ema.apply()
    torch.save({k: v.half() if v.dtype == torch.float32 else v for k, v in model.state_dict().items()},
               os.path.join(args.out, 'best.pt'))
    if ema:
        ema.restore()
    vo, to = evaluate('val'), evaluate('test')
    metrics = {'val': compute_metrics(vo['y'], vo['p']), 'test': compute_metrics(to['y'], to['p']),
               'n_params': n_params, 'n_router_params': n_router, 'final_val_auc': hist[-1]['val_auc'],
               'epochs_run': len(hist), 'minutes': (time.time() - t0) / 60}
    np.savez_compressed(os.path.join(args.out, 'test_outputs.npz'), **to)
    np.savez_compressed(os.path.join(args.out, 'val_outputs.npz'), **vo)
    if test_curve:
        np.savez_compressed(os.path.join(args.out, 'test_curve.npz'), p=np.stack(test_curve), y=to['y'])
    with open(os.path.join(args.out, 'history.csv'), 'w', newline='') as f:
        wr = csv.DictWriter(f, fieldnames=list(hist[0]))
        wr.writeheader()
        wr.writerows(hist)
    with open(os.path.join(args.out, 'config.json'), 'w') as f:
        json.dump(vars(args), f, indent=1)
    with open(os.path.join(args.out, 'metrics.json'), 'w') as f:   # written last: marks the run as finished
        json.dump(metrics, f, indent=1)
    print('DONE', {k: round(v, 4) for k, v in metrics['val'].items() if k in ('auc',)},
          'test', {k: round(v, 4) for k, v in metrics['test'].items() if k in ('auc', 'bench_acc', 'bench_f1')})


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/code/routed_vit/launcher.py
"""Run (variant, seed, fold) training jobs with one process per GPU (T4 x2), resumable (finished runs are skipped).

Jobs are ordered seed-major, then fold, then variant, so an interrupted session still leaves complete seeds behind.
"""
import os
import subprocess
import sys
import time

PKG_PARENT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def run_name(variant, seed, fold):
    return f'{variant}_s{seed}_f{fold}'


def run_jobs(variants, seeds, folds, samples, crops, runs_dir, n_gpus=2, extra=(), log_dir=None, per_variant_extra=None):
    """folds: iterable of fold ids (-1 = official train/val split). per_variant_extra: {variant: [extra args]} appended after `extra`."""
    log_dir = log_dir or os.path.join(runs_dir, 'logs')
    os.makedirs(log_dir, exist_ok=True)
    jobs = [(v, s, f) for s in seeds for f in folds for v in variants]
    running = {}   # gpu -> (Popen, name)
    env_base = dict(os.environ, PYTHONPATH=PKG_PARENT + os.pathsep + os.environ.get('PYTHONPATH', ''))
    n_gpus = max(1, n_gpus)
    while jobs or running:
        for gpu in range(n_gpus):
            if gpu not in running and jobs:
                v, s, f = jobs.pop(0)
                name = run_name(v, s, f)
                out = os.path.join(runs_dir, name)
                if os.path.exists(os.path.join(out, 'metrics.json')):
                    print(f'[skip] {name}')
                    continue
                cmd = [sys.executable, '-m', 'routed_vit.train', '--variant', v, '--seed', str(s), '--fold', str(f),
                       '--samples', samples, '--crops', crops, '--out', out, *map(str, extra),
                       *map(str, (per_variant_extra or {}).get(v, []))]
                env = dict(env_base, CUDA_VISIBLE_DEVICES=str(gpu))
                running[gpu] = (subprocess.Popen(cmd, stdout=open(os.path.join(log_dir, name + '.log'), 'w'),
                                                 stderr=subprocess.STDOUT, env=env, cwd=PKG_PARENT), name)
                print(f'[start] {name} on GPU {gpu}', flush=True)
        for gpu, (p, name) in list(running.items()):
            if p.poll() is not None:
                print(f'[done]  {name} exit={p.returncode}', flush=True)
                if p.returncode != 0:
                    print(open(os.path.join(log_dir, name + '.log')).read()[-3000:])
                del running[gpu]
        time.sleep(5)

In [ ]:
%%writefile /kaggle/working/code/routed_vit/analysis.py
"""Measurement suite (idea.md section 8). Everything that only needs saved test outputs works on CPU numpy; the model-based
probes (head importance 8.4, gradient share 8.5, swap tests 8.3) take a trained model and a CropDataset."""
import itertools
from math import comb

import numpy as np
import torch
import torch.nn.functional as F

from .data import prep


# ----------------------------------------------------------------------------- 8.2 routing structure
def entropy(p, axis=-1):
    return -(p * np.log(p + 1e-12)).sum(axis)


def routing_specialization(pi):
    """S^l = H(mean pi) - mean H(pi) = I(sample; head).  pi [n,Lr,H] -> S [Lr]."""
    return entropy(pi.mean(0)) - entropy(pi).mean(0)


def specialization_null(model, z, device, tau=1.0, seed=0):
    """Decorrelated-z reference for H2: permute every z feature independently across samples (keeps each feature's
    marginal, destroys the clip-specific structure), push through the trained routers, recompute S.
    NOTE: permuting whole z rows would leave S unchanged (S depends only on the multiset of pi)."""
    rng = np.random.default_rng(seed)
    zp = np.stack([rng.permutation(z[:, j]) for j in range(z.shape[1])], 1).astype(np.float32)
    zt = torch.from_numpy(zp).to(device)
    pis = []
    with torch.no_grad():
        for r in range(model.n_routed):
            _, pi, _, _ = model.routers[r].gate(zt, tau, True)
            pis.append(pi.cpu().numpy())
    return routing_specialization(np.stack(pis, 1))


def _codes(a):
    return np.unique(a, return_inverse=True)[1].reshape(-1)


def _H_mm(codes, n):
    c = np.bincount(codes)
    c = c[c > 0]
    p = c / n
    return -(p * np.log(p)).sum() + (len(c) - 1) / (2 * n)   # plug-in + Miller-Madow


def mi_mm(a, b):
    a, b = _codes(a), _codes(b)
    n = len(a)
    return _H_mm(a, n) + _H_mm(b, n) - _H_mm(a * (b.max() + 1) + b, n)


def cond_mi_mm(r, c, y):
    """I(r; c | y) = sum_y p(y) I(r; c | Y=y)."""
    tot = 0.0
    for v in np.unique(y):
        m = y == v
        if m.sum() > 1:
            tot += m.mean() * mi_mm(r[m], c[m])
    return tot


def route_ids(gates, mask=None, n_clusters=8, seed=0):
    """Discretised route per layer. B-sparse: the selected head set (one of C(H,k)); B-soft: k-means cluster of g.
    gates [n,Lr,H], mask [n,Lr,H] or None -> int [n,Lr]."""
    n, Lr, H = gates.shape
    out = np.zeros((n, Lr), dtype=np.int64)
    if mask is not None:
        out = (mask.astype(np.int64) * (2 ** np.arange(H))).sum(-1)
        return out
    from sklearn.cluster import KMeans
    for l in range(Lr):
        out[:, l] = KMeans(n_clusters=min(n_clusters, n), n_init=5, random_state=seed).fit_predict(gates[:, l])
    return out


def perm_ctx_by_ped(ctx, y, groups, rng):
    """Permute context among pedestrians within each label stratum (whole per-pedestrian c sequences are moved, tiled or
    truncated to the receiving pedestrian's window count), so within-pedestrian structure survives."""
    new = ctx.copy()
    idx_of = {}
    for i, g in enumerate(groups):
        idx_of.setdefault(g, []).append(i)
    for label in np.unique(y):
        peds = [g for g, ix in idx_of.items() if y[ix[0]] == label]
        donors = rng.permutation(len(peds))
        for p, d in zip(peds, donors):
            src = ctx[idx_of[peds[d]]]
            tgt = idx_of[p]
            new[tgt] = np.resize(src, (len(tgt), ctx.shape[1])) if len(src) != len(tgt) else src
    return new


def label_context_mi(route, y, ctx, ctx_names, groups, n_perm=1000, seed=0):
    """Per layer and context variable: I(r;y), I(r;c|y), permutation-null mean / 95th pct / p-value (H3)."""
    import pandas as pd
    rng = np.random.default_rng(seed)
    Lr = route.shape[1]
    perms = [perm_ctx_by_ped(ctx, y, groups, rng) for _ in range(n_perm)]
    rows = []
    for l in range(Lr):
        r = route[:, l]
        i_y = mi_mm(r, y)
        for k, name in enumerate(ctx_names):
            if len(np.unique(ctx[:, k])) < 2:
                continue
            obs = cond_mi_mm(r, ctx[:, k], y)
            null = np.array([cond_mi_mm(r, pc[:, k], y) for pc in perms])
            rows.append({'layer': l, 'context': name, 'I(r;y)': i_y, 'I(r;c|y)': obs, 'null_mean': null.mean(),
                         'null_p95': np.percentile(null, 95), 'p_value': (1 + (null >= obs).sum()) / (1 + n_perm),
                         'above_null': bool(obs > np.percentile(null, 95))})
    return pd.DataFrame(rows)


def gate_regression(gates, ctx, y, ctx_names):
    """Per layer and head: partial R^2 of context (all variables, one-hot) in  g ~ c + y.  -> [Lr,H]"""
    n, Lr, H = gates.shape
    cols = []
    for k in range(ctx.shape[1]):
        lv = np.unique(ctx[:, k])
        cols += [(ctx[:, k] == v).astype(float) for v in lv[1:]]
    Xc = np.stack(cols, 1) if cols else np.zeros((n, 0))
    Xy = np.stack([np.ones(n), y.astype(float)], 1)
    Xf = np.concatenate([Xy, Xc], 1)

    def r2(X, t):
        beta = np.linalg.lstsq(X, t, rcond=None)[0]
        v = t.var()
        return 0.0 if v < 1e-12 else 1 - (t - X @ beta).var() / v
    out = np.zeros((Lr, H))
    for l in range(Lr):
        for h in range(H):
            t = gates[:, l, h].astype(float)
            ry = r2(Xy, t)
            out[l, h] = (r2(Xf, t) - ry) / max(1 - ry, 1e-8)
    return out


# ----------------------------------------------------------------------------- model-based helpers
def _gather(ds, idxs):
    items = [ds[int(i)] for i in idxs]
    return torch.stack([b['x'] for b in items]), torch.stack([b['box'] for b in items])   # uint8 [n,T,3,H,W], [n,T,4]


@torch.no_grad()
def _logits(model, XB, device, bs=32, amp=True):
    model.eval()
    X, Bx = XB
    out = []
    for s in range(0, len(X), bs):
        x = prep(X[s:s + bs].to(device))
        with torch.autocast('cuda', dtype=torch.float16, enabled=amp and device.type == 'cuda'):
            out.append(model(x, box=Bx[s:s + bs].to(device))[0].float().cpu())
    return torch.cat(out).numpy()


# ----------------------------------------------------------------------------- 8.4 head importance (mean ablation)
@torch.no_grad()
def head_importance(model, ds, idxs, device, layers=None, bs=32, amp=True):
    """I[i,l,h] = |u(x_i) - u(x_i; O^{l,h} <- dataset-mean output of head)|.  Returns I [n,L,H], base logits [n]."""
    XB = _gather(ds, idxs)
    X = XB[0]
    L = len(model.blocks)
    layers = list(range(L)) if layers is None else list(layers)
    for l in layers:
        a = model.blocks[l].attn
        a.record = {'sum': torch.zeros(a.h, 197, a.dk, device=device), 'n': 0}
    base = _logits(model, XB, device, bs, amp)
    means = {}
    for l in layers:
        a = model.blocks[l].attn
        means[l] = a.record['sum'] / a.record['n']
        a.record = None
    imp = np.zeros((len(X), L, model.H), dtype=np.float32)
    for l in layers:
        for h in range(model.H):
            model.blocks[l].attn.ablate = (h, means[l][h])
            imp[:, l, h] = np.abs(base - _logits(model, XB, device, bs, amp))
        model.blocks[l].attn.ablate = None
    return imp, base


# ----------------------------------------------------------------------------- 8.5 gradient share
def gradient_norms(model, ds, idxs, device, layers):
    """||grad_{theta^{l,h}} loss_i|| for theta^{l,h} = (Q,K,V rows of head h, W_O columns of head h). -> [n,len(layers),H]"""
    model.eval()
    H, dk = model.H, model.blocks[0].attn.dk
    params = [(model.blocks[l].attn.qkv.weight, model.blocks[l].attn.qkv.bias, model.blocks[l].attn.proj.weight)
              for l in layers]
    flat = [p for t in params for p in t]
    out = np.zeros((len(idxs), len(layers), H), dtype=np.float32)
    for n, i in enumerate(idxs):
        b = ds[int(i)]
        x = prep(b['x'][None].to(device))
        loss = F.binary_cross_entropy_with_logits(model(x, box=b['box'][None].to(device))[0], b['y'].to(device)[None])
        grads = torch.autograd.grad(loss, flat, allow_unused=True)
        for li in range(len(layers)):
            gq, gb, go = [g if g is not None else torch.zeros_like(p) for g, p in zip(grads[3 * li:3 * li + 3], flat[3 * li:3 * li + 3])]
            d = H * dk
            for h in range(H):
                rows = [c * d + h * dk + j for c in range(3) for j in range(dk)]
                sq = gq[rows].pow(2).sum() + gb[rows].pow(2).sum() + go[:, h * dk:(h + 1) * dk].pow(2).sum()
                out[n, li, h] = float(sq.sqrt())
    return out


def gradient_share_entropy(norms, ctx_col, normalise_by_freq=True):
    """q^l_h(c) = sum_{i:c_i=c} ||grad|| / sum_i ||grad||;  returns per-head normalised entropy H(q)/log(#bins) [Lr,H].
    With normalise_by_freq the per-bin MEAN norm is used, so that unbalanced context bins do not look 'specialised'."""
    bins = np.unique(ctx_col)
    q = np.stack([(norms[ctx_col == b].mean(0) if normalise_by_freq else norms[ctx_col == b].sum(0)) for b in bins])
    q = q / (q.sum(0, keepdims=True) + 1e-12)
    return entropy(q, axis=0) / np.log(max(len(bins), 2))


# ----------------------------------------------------------------------------- 8.3 causal route tests
@torch.no_grad()
def _routed_logits(model, tok0_unique, inv, gates_pairs, T, amp, device, box):
    """tok0_unique [U*T,N,d]; inv [P] row of the unique sample for each pair; gates_pairs [P,Lr,H] -> logits, xL."""
    U = tok0_unique.shape[0] // T
    tok = tok0_unique.view(U, T, *tok0_unique.shape[1:])[torch.as_tensor(inv, device=device)].flatten(0, 1)
    g = [gates_pairs[:, r].float() for r in range(gates_pairs.shape[1])]
    with torch.autocast('cuda', dtype=torch.float16, enabled=amp and device.type == 'cuda'):
        logit, info = model.routed_forward(tok, len(inv), T, gates=g, box=box)
    return logit.float().cpu().numpy(), info['xL'].cpu().numpy()


@torch.no_grad()
def swap_experiment(model, ds, own_gates, y, ctx, ctx_names, device, n_pairs=40, n_bins=5, n_ctrl=8, n_cand=40000,
                    bs=32, amp=True, seed=0, tol=0.10):
    """Swap effect  Delta_ij = |u(x_i; g_j) - u(x_i; g_i)|  with pairs matched on route distance ||g_i-g_j||_1, plus the
    magnitude-matched random-gate control (random gate = a real gate with heads permuted per layer, best of n_ctrl candidates
    whose residual-stream change ||X^L(g')-X^L(g_i)|| is closest to the real swap's; matched if within +-tol).
    Returns a DataFrame with one row per pair."""
    import pandas as pd
    rng = np.random.default_rng(seed)
    model.eval()
    n, Lr, H = own_gates.shape
    G = own_gates.reshape(n, -1)
    pairs = {}   # (i,j) -> set of (context, group)
    for k, name in enumerate(ctx_names):
        c = ctx[:, k]
        if len(np.unique(c)) < 2:
            continue
        i = rng.integers(0, n, n_cand)
        j = rng.integers(0, n, n_cand)
        ok = i != j
        i, j = i[ok], j[ok]
        dist = np.abs(G[i] - G[j]).sum(1)
        g1 = (y[i] == y[j]) & (c[i] != c[j])     # same label, different context
        g2 = (c[i] == c[j]) & (y[i] != y[j])     # same context, different label
        if g1.sum() < n_bins or g2.sum() < n_bins:
            continue
        edges = np.quantile(dist[g1 | g2], np.linspace(0, 1, n_bins + 1))
        b = np.clip(np.searchsorted(edges, dist, side='right') - 1, 0, n_bins - 1)
        for bb in range(n_bins):
            m1, m2 = np.where(g1 & (b == bb))[0], np.where(g2 & (b == bb))[0]
            m = min(len(m1), len(m2), n_pairs // n_bins)
            for grp, mm in ((1, m1), (2, m2)):
                for q in rng.choice(mm, m, replace=False) if m > 0 else []:
                    pairs.setdefault((int(i[q]), int(j[q])), set()).add((name, grp))
    plist = sorted(pairs)
    if not plist:
        return pd.DataFrame()
    ids = sorted({i for i, _ in plist})
    X, BX = _gather(ds, ids)
    pos = {i: r for r, i in enumerate(ids)}
    g_t = torch.from_numpy(own_gates).float().to(device)
    res = {k: [] for k in ['u_own', 'u_swap', 'xL_own', 'xL_swap', 'u_ctrl', 'mag_ctrl']}
    T = X.shape[1]
    for s in range(0, len(plist), bs):
        chunk = plist[s:s + bs]
        u_ids = sorted({i for i, _ in chunk})
        tok0 = model.embed_to_L0(prep(X[[pos[i] for i in u_ids]].to(device)))
        inv = [u_ids.index(i) for i, _ in chunk]
        box = BX[[pos[i] for i, _ in chunk]].to(device)   # the pair's own box track (route is swapped, box is not)
        gi = g_t[[i for i, _ in chunk]]
        gj = g_t[[j for _, j in chunk]]
        u_own, x_own = _routed_logits(model, tok0, inv, gi, T, amp, device, box)
        u_sw, x_sw = _routed_logits(model, tok0, inv, gj, T, amp, device, box)
        mag = np.linalg.norm(x_sw - x_own, axis=1)
        best_u, best_mag = np.zeros(len(chunk)), np.full(len(chunk), np.nan)
        best_gap = np.full(len(chunk), np.inf)
        for _ in range(n_ctrl):
            donors = g_t[torch.from_numpy(rng.integers(0, n, len(chunk))).to(device)]
            perm = torch.rand(len(chunk), Lr, H, device=device).argsort(-1)
            gc = torch.gather(donors, 2, perm)
            u_c, x_c = _routed_logits(model, tok0, inv, gc, T, amp, device, box)
            mc = np.linalg.norm(x_c - x_own, axis=1)
            gap = np.abs(mc - mag) / (mag + 1e-8)
            better = gap < best_gap
            best_gap = np.where(better, gap, best_gap)
            best_u = np.where(better, u_c, best_u)
            best_mag = np.where(better, mc, best_mag)
        res['u_own'].append(u_own)
        res['u_swap'].append(u_sw)
        res['xL_own'].append(mag)
        res['xL_swap'].append(best_gap)
        res['u_ctrl'].append(best_u)
        res['mag_ctrl'].append(best_mag)
    r = {k: np.concatenate(v) for k, v in res.items()}
    delta = np.abs(r['u_swap'] - r['u_own'])
    delta_c = np.abs(r['u_ctrl'] - r['u_own'])
    rows = []
    for q, (i, j) in enumerate(plist):
        for name, grp in pairs[(i, j)]:
            rows.append({'context': name, 'group': 'ctx_differs' if grp == 1 else 'label_differs', 'i': i, 'j': j,
                         'delta': delta[q], 'delta_ctrl': delta_c[q], 'mag': r['xL_own'][q], 'matched': r['xL_swap'][q] <= tol})
    return pd.DataFrame(rows)


def swap_summary(df, n_boot=1000, seed=0):
    """Gamma = E[Delta | y_i=y_j, c_i!=c_j] - E[Delta | c_i=c_j, y_i!=y_j] per context; also the 'kept' version
    (only swaps with Delta > Delta' and a matched control) and the same contrast computed on the controls."""
    import pandas as pd
    rng = np.random.default_rng(seed)
    rows = []
    for name, d in df.groupby('context'):
        a, b = d[d.group == 'ctx_differs'], d[d.group == 'label_differs']
        if len(a) < 3 or len(b) < 3:
            continue

        def gam(a, b, col):
            return a[col].mean() - b[col].mean()
        boots = [gam(a.iloc[rng.integers(0, len(a), len(a))], b.iloc[rng.integers(0, len(b), len(b))], 'delta')
                 for _ in range(n_boot)]
        ka, kb = a[(a.delta > a.delta_ctrl) & a.matched], b[(b.delta > b.delta_ctrl) & b.matched]
        rows.append({'context': name, 'n_ctx_differs': len(a), 'n_label_differs': len(b),
                     'Gamma': gam(a, b, 'delta'), 'Gamma_lo': np.percentile(boots, 2.5), 'Gamma_hi': np.percentile(boots, 97.5),
                     'Gamma_ctrl': gam(a, b, 'delta_ctrl'),
                     'frac_swap_gt_ctrl': float(((d.delta > d.delta_ctrl) & d.matched).sum() / max(d.matched.sum(), 1)),
                     'Gamma_kept': gam(ka, kb, 'delta') if len(ka) >= 3 and len(kb) >= 3 else np.nan,
                     'n_kept': len(ka) + len(kb)})
    return pd.DataFrame(rows)


def n_routes(H=6, k=3):
    return comb(H, k)

In [ ]:
%%writefile /kaggle/working/code/routed_vit/reports.py
"""Notebook-level helpers: load trained runs, result tables with CIs, and the per-run analysis drivers + plots."""
import json
import os

import numpy as np
import pandas as pd
import torch

from . import analysis as an
from .data import CTX_LEVELS, CTX_NAMES, CropDataset, groups_of
from .metrics import bootstrap_metrics
from .model import RoutedViT


def load_run(run_dir, device):
    cfg = json.load(open(os.path.join(run_dir, 'config.json')))
    m = RoutedViT(cfg['variant'], L0=cfg['L0'], k=cfg['k'], use_box=not cfg.get('no_box', False),
                  box_only=cfg.get('box_only', False))
    sd = torch.load(os.path.join(run_dir, 'best.pt'), map_location='cpu')
    m.load_state_dict({k: v.float() if v.dtype == torch.float16 else v for k, v in sd.items()})
    return m.to(device).eval(), cfg


def load_outputs(run_dir, split='test'):
    return dict(np.load(os.path.join(run_dir, f'{split}_outputs.npz')))


def rdir(root, variant, seed, fold):
    return os.path.join(root, f'{variant}_s{seed}_f{fold}')


def is_done(root, variant, seed, fold):
    return os.path.exists(os.path.join(rdir(root, variant, seed, fold), 'metrics.json'))


def collect(root, variant, seeds, folds, n_pool, n_test):
    """Per training seed: out-of-fold probabilities over the pooled train+val record (NaN where a fold is missing) and the
    fold-ensembled (mean) test probabilities. -> {seed: {'oof': [n_pool], 'test': [n_test], 'folds_done': int, 'fold_auc': [..]}}"""
    from sklearn.metrics import roc_auc_score
    res = {}
    for s in seeds:
        oof, tst, k, fold_auc = np.full(n_pool, np.nan), np.zeros(n_test), 0, []
        for f in folds:
            if not is_done(root, variant, s, f):
                continue
            d = rdir(root, variant, s, f)
            vo, to = load_outputs(d, 'val'), load_outputs(d, 'test')
            oof[vo['idx']] = vo['p']
            fold_auc.append(roc_auc_score(vo['y'], vo['p']))
            tst[to['idx']] += to['p']
            k += 1
        if k:
            res[s] = {'oof': oof, 'test': tst / k, 'folds_done': k, 'fold_auc': fold_auc}
    return res


def results_table(root, variants, seeds, folds, data, n_boot=500):
    """Cross-validated results. Per (variant, seed): out-of-fold AUC on the pooled 216 train+val pedestrians (selection-free:
    every window is predicted by a model that never saw its video), per-fold AUC mean +- std, and the test AUC of the fold
    ensemble. The decision threshold for acc/F1 is picked on the OUT-OF-FOLD predictions (balanced accuracy), never on test.
    A final row per variant ('ens') averages all seeds and folds, with pedestrian-cluster bootstrap CIs on test."""
    from sklearn.metrics import roc_auc_score
    from .cv import pool_trainval
    from .metrics import compute_metrics, select_threshold
    pool = pool_trainval(data)
    y_pool, y_te = np.asarray(pool['label']), np.asarray(data['test']['label'])
    g_te = groups_of(data['test'])
    maj = max(y_te.mean(), 1 - y_te.mean())
    rows = []
    for v in variants:
        col = collect(root, v, seeds, folds, len(y_pool), len(y_te))
        for s, c in col.items():
            full = c['folds_done'] == len(folds)
            m_oof = ~np.isnan(c['oof'])
            thr = select_threshold(y_pool[m_oof], c['oof'][m_oof]) if full else 0.5
            mt = compute_metrics(y_te, c['test'], thr)
            rows.append({'variant': v, 'seed': s, 'folds': c['folds_done'],
                         'oof_auc': roc_auc_score(y_pool[m_oof], c['oof'][m_oof]),
                         'fold_auc': f"{np.mean(c['fold_auc']):.3f}±{np.std(c['fold_auc']):.3f}",
                         'test_auc': mt['auc'], 'bench_acc': mt['bench_acc'], 'bench_f1': mt['bench_f1'],
                         'thr': thr, 'thr_bacc': mt['thr_balanced_acc'], 'thr_f1': mt['thr_f1'], 'bench_auc': mt['bench_auc'],
                         'brier': mt['brier']})
        if col:
            p_ens = np.mean([c['test'] for c in col.values()], 0)
            oof_ens = np.nanmean([c['oof'] for c in col.values()], 0)
            m_oof = ~np.isnan(oof_ens)
            thr = select_threshold(y_pool[m_oof], oof_ens[m_oof])
            ci = bootstrap_metrics(y_te, p_ens, g_te, thr, n_boot=n_boot, keys=('auc', 'bench_acc', 'bench_f1', 'thr_f1'))
            mt = compute_metrics(y_te, p_ens, thr)
            rows.append({'variant': v, 'seed': 'ens', 'folds': sum(c['folds_done'] for c in col.values()),
                         'oof_auc': roc_auc_score(y_pool[m_oof], oof_ens[m_oof]), 'fold_auc': '',
                         'test_auc': mt['auc'], 'bench_acc': mt['bench_acc'], 'bench_f1': mt['bench_f1'], 'thr': thr,
                         'thr_bacc': mt['thr_balanced_acc'], 'thr_f1': mt['thr_f1'], 'bench_auc': mt['bench_auc'],
                         'brier': mt['brier'], 'test_auc_ci': '[%.3f, %.3f]' % ci['auc'][1:],
                         'bench_acc_ci': '[%.3f, %.3f]' % ci['bench_acc'][1:], 'thr_f1_ci': '[%.3f, %.3f]' % ci['thr_f1'][1:]})
    df = pd.DataFrame(rows)
    print(f'test majority-class accuracy {maj:.3f} (positive rate {y_te.mean():.3f}): acc/F1@0.5 below that carry no signal; read AUC.')
    if not df.empty:
        per = df[df.seed != 'ens']
        print('mean ± std over training seeds:')
        print(per.groupby('variant')[['oof_auc', 'test_auc', 'thr_bacc', 'thr_f1']].agg(['mean', 'std']).round(3).to_string())
    return df


def ensemble_test_probs(root, variant, seeds, folds, n_pool, n_test):
    """One fold-ensembled test-probability vector per finished seed (input of the paired H1 bootstrap)."""
    col = collect(root, variant, seeds, folds, n_pool, n_test)
    return [c['test'] for c in col.values() if c['folds_done'] == len(folds)]


def plot_history(root, variants, seeds, folds):
    """Mean +- min/max over folds and seeds of the per-epoch validation AUC (EMA model) and training loss."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for v in variants:
        hs = [pd.read_csv(os.path.join(rdir(root, v, s, f), 'history.csv')) for s in seeds for f in folds if is_done(root, v, s, f)]
        if not hs:
            continue
        for a, col in zip(ax, ['val_auc', 'train_task']):
            M = np.stack([h[col].values for h in hs])
            x = hs[0].epoch.values
            a.plot(x, M.mean(0), label=f'{v} (n={len(hs)})')
            a.fill_between(x, M.min(0), M.max(0), alpha=.15)
    ax[0].set_title('held-out-fold AUC (mean, min-max band)')
    ax[1].set_title('train loss (task)')
    ax[0].legend(fontsize=7)
    for a in ax:
        a.set_xlabel('epoch')
    plt.tight_layout()
    plt.show()


def heatmap(mat, title, xlabel='head', ylabel='layer', ax=None, ylabels=None, vmin=None, vmax=None):
    import matplotlib.pyplot as plt
    ax = ax or plt.gca()
    im = ax.imshow(mat, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(mat.shape[1]))
    if ylabels is not None:
        ax.set_yticks(range(mat.shape[0]))
        ax.set_yticklabels(ylabels)
    plt.colorbar(im, ax=ax)


# --------------------------------------------------------------------------- Phase A / H5 probes (any variant)
def head_probes(run_dir, data, crops, device, n_samples=300, seed=0, importance=True):
    """Head importance (mean ablation) and gradient norms on a fixed random test subset; saved to <run_dir>/probes.npz."""
    out = os.path.join(run_dir, 'probes.npz')
    if os.path.exists(out):
        return dict(np.load(out))
    model, cfg = load_run(run_dir, device)
    ds = CropDataset(data['test'], crops)
    rng = np.random.default_rng(seed)
    idxs = np.sort(rng.choice(len(ds), min(n_samples, len(ds)), replace=False))
    routed_layers = list(range(model.L0, len(model.blocks)))
    res = {'idxs': idxs, 'grad_norms': an.gradient_norms(model, ds, idxs, device, routed_layers)}
    if importance:
        imp, base = an.head_importance(model, ds, idxs, device)
        res.update(importance=imp, base_logit=base)
    np.savez_compressed(out, **res)
    del model
    torch.cuda.empty_cache()
    return res


def grad_entropy_table(probe, ctx_test, names=CTX_NAMES):
    """Mean (over routed layers x heads) normalised entropy of the gradient share per context variable (H5)."""
    ctx = ctx_test[probe['idxs']]
    rows = {}
    for k, n in enumerate(names):
        if len(np.unique(ctx[:, k])) < 2:
            continue
        rows[n] = float(an.gradient_share_entropy(probe['grad_norms'], ctx[:, k]).mean())
    return rows


def importance_by_context(probe, ctx_test, name):
    """Mean head importance per context level: dict level -> [L,H]."""
    k = CTX_NAMES.index(name)
    ctx = ctx_test[probe['idxs']][:, k]
    return {CTX_LEVELS[name][b]: probe['importance'][ctx == b].mean(0) for b in np.unique(ctx)}

## 2. Config

In [ ]:
import os, sys, json, glob, shutil, pickle, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
WORK = '/kaggle/working'
sys.path.insert(0, f'{WORK}/code')
REPO, JAAD, OUT = f'{WORK}/repo', f'{WORK}/JAAD', f'{WORK}/out'
RUNS = f'{OUT}/runs'
SAMPLE_TYPE = 'beh'            # 'beh' (JAAD_beh, default) or 'all' (JAAD_all, ~4x more samples/crops)
SEEDS = [42, 7, 97]            # training seeds
N_FOLDS = 5                    # video-grouped CV folds over train+val (216 pedestrians); the official test split is only predicted
FOLDS = list(range(N_FOLDS))
MICRO_BS, WORKERS = 16, 2      # per GPU process; 16 clips x 8 frames = 128 images / step (lower MICRO_BS on OOM)
BASE_EXTRA = ['--n_folds', N_FOLDS, '--micro_bs', MICRO_BS, '--workers', WORKERS]
CROPS = f'{WORK}/crops_{SAMPLE_TYPE}'
SAMPLES = f'{OUT}/jaad_{SAMPLE_TYPE}.pkl'
os.makedirs(RUNS, exist_ok=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('GPUs:', torch.cuda.device_count(), '| samples:', SAMPLES)

VARIANTS = ['soft', 'sparse', 'film']
ANA_FOLDS = [0, 1]         # runs (per seed) used for the H2/H3 analyses; probes (H5) and swap tests (H4) use fold 0 only
SWAP_SEEDS = [42]          # swap tests are the expensive probe; add seeds if time allows
N_PERM = 1000              # permutation null size (idea.md 8.2)
# notebook A's output (runs + probes + recipe), if attached
A_DIRS = glob.glob('/kaggle/input/**/out', recursive=True)
A_OUT = next((d for d in A_DIRS if glob.glob(d + '/runs/dense_s*')), None)
A_RUNS = A_OUT + '/runs' if A_OUT else None
print('notebook-A output:', A_OUT)

## 3. Data (JAAD, same PCPA protocol as the base repo)
Samples = 16 observed frames, TTE 30-60, stride 3, official video split. A clip uses 8 of the 16 frames (odd indices, last frame included) as 224x224 pedestrian-centred square crops (1.5x box). Annotated context (never a model input) is read from the JAAD annotations.

In [ ]:
# JAAD annotations + split ids + jaad_data.py; the 346 mp4s come from the attached Kaggle dataset (no download), fallback = official zip
clips = glob.glob('/kaggle/input/**/JAAD_clips', recursive=True)
cmd = f'--clips_from {clips[0]}' if clips else '--clips'
print('clips source:', clips[0] if clips else 'downloading official zip (3.1 GB)')
!python {REPO}/download_jaad.py --root {JAAD} {cmd}

In [ ]:
from routed_vit.data import build_samples, load_samples, context_table
# reuse samples/crops from an attached earlier notebook output if available
prev = glob.glob(f'/kaggle/input/**/out/jaad_{SAMPLE_TYPE}.pkl', recursive=True)
if os.path.exists(SAMPLES):
    data = load_samples(SAMPLES)
elif prev:
    shutil.copy(prev[0], SAMPLES); data = load_samples(SAMPLES); print('reused', prev[0])
else:
    data = build_samples(REPO, JAAD, SAMPLES, SAMPLE_TYPE)
for s in ['train', 'val', 'test']:
    r = data[s]; print(f"{s}: windows={len(r['label'])} peds={len(set(zip(r['vid'], r['pid'])))} pos={int(r['label'].sum())}")
context_table(data)

In [ ]:
from routed_vit.data import build_crop_cache, CropDataset
prev = glob.glob(f'/kaggle/input/**/crops_{SAMPLE_TYPE}', recursive=True)
if prev and not os.path.exists(CROPS):
    CROPS = prev[0]; print('reusing crop cache', CROPS)
else:
    build_crop_cache(data, f'{JAAD}/JAAD_clips', CROPS, workers=4)
print('crop cache:', CROPS, f'{sum(os.path.getsize(os.path.join(CROPS, f)) for f in os.listdir(CROPS)) / 1e9:.2f} GB')

In [ ]:
# look at what the model sees: 8 crops of 3 training clips (pos / neg)
ds = CropDataset(data['train'], CROPS)
fig, ax = plt.subplots(3, 8, figsize=(16, 6.4))
for r, i in enumerate(np.random.default_rng(0).choice(len(ds), 3, replace=False)):
    b = ds[int(i)]
    for t in range(8):
        ax[r, t].imshow(b['x'][t].permute(1, 2, 0).numpy()); ax[r, t].axis('off')
    ax[r, 0].set_title(f"label={int(b['y'])}", fontsize=8, loc='left')
plt.tight_layout(); plt.show()

### Cross-validation folds (identical to notebook A)

In [ ]:
from routed_vit.cv import pool_trainval, make_folds, describe_folds
pool = pool_trainval(data)
N_POOL, N_TEST = len(pool['label']), len(data['test']['label'])
display(pd.DataFrame(describe_folds(pool, make_folds(pool, N_FOLDS, 0))))

### Fine-tuning recipe (selected in notebook A by pooled out-of-fold AUC)

In [ ]:
from routed_vit.recipes import RECIPES, DEFAULT_RECIPE, SPARSE_EXTRA
if A_OUT and os.path.exists(A_OUT + '/recipe.json'):
    RECIPE = json.load(open(A_OUT + '/recipe.json'))['recipe']
else:
    RECIPE = DEFAULT_RECIPE; print('recipe.json not found: falling back to the default recipe (attach notebook A output to use its selection)')
print('recipe:', RECIPE, RECIPES[RECIPE])

## 4. Sanity checks: Prop. 1 (B nests A), parameter overhead, memory

In [ ]:
from routed_vit.model import RoutedViT, count_params
from routed_vit.data import prep
torch.manual_seed(0); A = RoutedViT('dense', drop_path=0.1).to(device).eval()
torch.manual_seed(0); B = RoutedViT('soft', drop_path=0.1).to(device).eval()
x = prep(torch.randint(0, 255, (4, 8, 3, 224, 224), dtype=torch.uint8, device=device)); bx = torch.randn(4, 8, 4, device=device)
with torch.no_grad():
    ua, ub = A(x, bx)[0], B(x, bx)[0]
print('Prop.1  max |u_A - u_Bsoft| at init =', float((ua - ub).abs().max()), '(must be 0)')
for v in ['dense', 'soft', 'sparse', 'film']:
    tot, r = count_params(RoutedViT(v)); print(f'{v:7s} params {tot / 1e6:.3f}M  routing/film params {r / 1e3:.1f}k')
del A, B
m = RoutedViT('sparse', drop_path=0.1).to(device).train()
xx = prep(torch.randint(0, 255, (MICRO_BS, 8, 3, 224, 224), dtype=torch.uint8, device=device), train=True)
with torch.autocast('cuda', dtype=torch.float16):
    l, info = m(xx, box=torch.randn(MICRO_BS, 8, 4, device=device))
(l.float().mean() + info['bal'] + info['zloss']).backward()
print('B-sparse peak GPU mem %.1f GB at %d clips/step' % (torch.cuda.max_memory_allocated() / 1e9, MICRO_BS))
del m, xx, l, info; torch.cuda.empty_cache()

## 5. Train B variants (same recipe, folds and seeds as A)

In [ ]:
from routed_vit.launcher import run_jobs
run_jobs(VARIANTS, SEEDS, FOLDS, SAMPLES, CROPS, RUNS, n_gpus=torch.cuda.device_count() or 1,
         extra=BASE_EXTRA + RECIPES[RECIPE], per_variant_extra={'sparse': SPARSE_EXTRA})

## 6. H1 — does routing help? (fold-ensembled test AUC, paired pedestrian-cluster bootstrap vs A)

In [ ]:
from routed_vit.reports import results_table, plot_history, load_outputs, ensemble_test_probs
from routed_vit.metrics import paired_auc_diff
from routed_vit.data import groups_of
plot_history(RUNS, VARIANTS, SEEDS, FOLDS)
res_B = results_table(RUNS, VARIANTS, SEEDS, FOLDS, data)
display(res_B[[c for c in ['variant', 'seed', 'folds', 'oof_auc', 'fold_auc', 'test_auc', 'test_auc_ci', 'bench_acc', 'bench_f1', 'thr', 'thr_bacc', 'thr_f1', 'bench_auc'] if c in res_B]])
y_te, grp_te = data['test']['label'], groups_of(data['test'])
h1 = {}
if A_RUNS:
    pA = ensemble_test_probs(A_RUNS, 'dense', SEEDS, FOLDS, N_POOL, N_TEST)
    res_A = results_table(A_RUNS, ['dense'], SEEDS, FOLDS, data)
    for v in VARIANTS:
        pB = ensemble_test_probs(RUNS, v, SEEDS, FOLDS, N_POOL, N_TEST)
        if pA and pB:
            h1[v] = paired_auc_diff(y_te, pA, pB, grp_te)
            print(f'H1  AUC({v}) - AUC(A) = {h1[v][0]:+.4f}  95% CI [{h1[v][1]:+.4f}, {h1[v][2]:+.4f}]  ->', 'supported' if h1[v][1] > 0 else 'not supported')
    display(res_A[[c for c in ['variant', 'seed', 'folds', 'oof_auc', 'fold_auc', 'test_auc', 'test_auc_ci', 'bench_acc', 'bench_f1', 'thr', 'thr_bacc', 'thr_f1', 'bench_auc'] if c in res_A]])
else:
    print('Attach notebook A output to compute H1 (paired comparison against the dense model).')

## 7. H2/H3 — is the routing structured, and does it follow context beyond the label?
$\mathcal S^l=I(\text{sample};\text{head})$ against a decorrelated-$z$ reference (each $z$ feature permuted independently; permuting whole rows would leave $\mathcal S$ unchanged). Routes $r$: selected head set (B-sparse) or 8-means cluster of $g$ (B-soft). $I(r;c\mid y)$ with Miller-Madow correction vs. a permutation null that shuffles context between pedestrians within each label stratum.
**H2 pass rule (tightened):** $\mathcal S$ above the reference in more than half of the cells **and** mean $\mathcal S/\log H > 0.05$. Round 1 passed the first condition with $\mathcal S/\log H \approx 0.001$, which is practically collapsed.
The first cell also reports how many distinct routes (head subsets) B-sparse actually uses per layer (out of 20 possible).

In [ ]:
from routed_vit import analysis as an
from routed_vit.reports import load_run, heatmap
from routed_vit.data import CTX_NAMES
ctx_te = data['test']['ctx']
ANA = [(v, s, f) for v in ['soft', 'sparse'] for s in SEEDS for f in ANA_FOLDS if os.path.exists(f'{RUNS}/{v}_s{s}_f{f}/metrics.json')]
H2, H3, USED = [], [], []
for v, s, f in ANA:
    d = f'{RUNS}/{v}_s{s}_f{f}'
    o = load_outputs(d); idx = o['idx']
    S = an.routing_specialization(o['pi'])
    model, _ = load_run(d, device)
    null = np.mean([an.specialization_null(model, o['z'].astype(np.float32), device, seed=k) for k in range(10)], 0)
    del model
    H2.append(pd.DataFrame({'variant': v, 'seed': s, 'fold': f, 'layer': range(7, 13), 'S': S, 'S_null': null, 'S/logH': S / np.log(6)}))
    route = an.route_ids(o['gates'], o.get('mask'))
    if v == 'sparse':
        USED.append({'variant': v, 'seed': s, 'fold': f, 'distinct_routes_per_layer': [int(len(np.unique(route[:, l]))) for l in range(route.shape[1])]})
    t = an.label_context_mi(route, data['test']['label'][idx], ctx_te[idx], CTX_NAMES, grp_te[idx], n_perm=N_PERM)
    t.insert(0, 'fold', f); t.insert(0, 'seed', s); t.insert(0, 'variant', v); H3.append(t)
H2 = pd.concat(H2); H3 = pd.concat(H3)
display(H2.groupby(['variant', 'layer'])[['S', 'S_null', 'S/logH']].mean().round(4))
display(pd.DataFrame(USED))

In [ ]:
# H3: per context variable, in how many (seed, fold, layer) cells is I(r;c|y) above the 95th percentile of the permutation null?
H3['above'] = H3['above_null']
tab = H3.groupby(['variant', 'context']).agg(cells=('above', 'size'), above=('above', 'sum'), I_r_y=('I(r;y)', 'mean'),
                                              I_r_c_given_y=('I(r;c|y)', 'mean'), null_p95=('null_p95', 'mean'), min_p=('p_value', 'min'))
display(tab.round(4))
# gate regression: partial R^2 of context (all variables) in g ~ c + y, per layer x head
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, v in zip(ax, ['soft', 'sparse']):
    rs = []
    for vv, s, f in ANA:
        if vv != v: continue
        o = load_outputs(f'{RUNS}/{v}_s{s}_f{f}')
        rs.append(an.gate_regression(o['gates'], ctx_te[o['idx']], data['test']['label'][o['idx']], CTX_NAMES))
    if rs: heatmap(np.mean(rs, 0), f'{v}: partial R2 of context in gate', ax=a, ylabels=range(7, 13), vmin=0)
plt.tight_layout(); plt.show()

## 8. H4 — causal route tests (swap the gates of clip $j$ into clip $i$)
$\Gamma=E[\Delta\mid y_i=y_j,c_i\ne c_j]-E[\Delta\mid c_i=c_j,y_i\ne y_j]$ with pairs matched on $\|g_i-g_j\|_1$; control = random gate (a real gate with heads shuffled per layer) matched to the same residual-stream change (+-10%). `frac_swap_gt_ctrl` = share of matched swaps whose effect exceeds the control; `Gamma_kept` uses only those. Fold 0 of the seeds in `SWAP_SEEDS`.

In [ ]:
from routed_vit.data import CropDataset
H4 = []
ds_te = CropDataset(data['test'], CROPS)
for v in ['soft', 'sparse']:
    for s in SWAP_SEEDS:
        d = f'{RUNS}/{v}_s{s}_f0'
        if not os.path.exists(d + '/metrics.json'): continue
        model, _ = load_run(d, device); o = load_outputs(d)
        pairs = an.swap_experiment(model, ds_te, o['gates'][np.argsort(o['idx'])], data['test']['label'], ctx_te, CTX_NAMES, device, n_pairs=40, n_ctrl=8)
        del model; torch.cuda.empty_cache()
        t = an.swap_summary(pairs); t.insert(0, 'seed', s); t.insert(0, 'variant', v); H4.append(t)
        pairs.to_csv(f'{OUT}/swap_pairs_{v}_s{s}.csv', index=False)
H4 = pd.concat(H4) if H4 else pd.DataFrame()
display(H4.round(4))

## 9. H5 — does selection drive specialization? (normalised entropy of the gradient share over context bins, lower = more specialised)

In [ ]:
from routed_vit.reports import head_probes, grad_entropy_table
ent = {}
for v in ['soft', 'sparse']:
    for s in SEEDS:
        if os.path.exists(f'{RUNS}/{v}_s{s}_f0/metrics.json'):
            ent[(v, s)] = grad_entropy_table(head_probes(f'{RUNS}/{v}_s{s}_f0', data, CROPS, device, n_samples=300, importance=False), ctx_te)
if A_RUNS:
    for s in SEEDS:
        if os.path.exists(f'{A_RUNS}/dense_s{s}_f0/probes.npz'):
            ent[('dense', s)] = grad_entropy_table(dict(np.load(f'{A_RUNS}/dense_s{s}_f0/probes.npz')), ctx_te)
E = pd.DataFrame(ent).T
E.index.names = ['variant', 'seed']
H5 = E.groupby('variant').mean(); H5['mean_over_contexts'] = H5.mean(1)
display(H5.round(3))

## 10. Hypotheses summary

In [ ]:
rows = []
for v in VARIANTS:
    if v in h1: rows.append({'hyp': 'H1', 'variant': v, 'statistic': f'dAUC={h1[v][0]:+.4f} [{h1[v][1]:+.4f},{h1[v][2]:+.4f}]', 'verdict': 'supported' if h1[v][1] > 0 else 'not supported'})
for v, d in H2.groupby('variant'):
    f = float((d.S > d.S_null).mean()); r = d['S/logH'].mean()
    rows.append({'hyp': 'H2', 'variant': v, 'statistic': f'S>S_null in {f:.0%} of cells; mean S/logH={r:.3f}', 'verdict': 'supported' if (f > 0.5 and r > 0.05) else 'not supported (collapsed)'})
for v, d in H3.groupby('variant'):
    f = float(d.above_null.mean()); rows.append({'hyp': 'H3', 'variant': v, 'statistic': f'I(r;c|y) > null p95 in {f:.0%} of (seed,fold,layer,context) cells', 'verdict': 'supported' if f > 0.25 else 'not supported'})
if len(H4):
    for v, d in H4.groupby('variant'):
        ok = ((d.Gamma_lo > 0) & (d.frac_swap_gt_ctrl > 0.5)).mean(); rows.append({'hyp': 'H4', 'variant': v, 'statistic': f'Gamma CI>0 and swaps>control for {ok:.0%} of contexts', 'verdict': 'supported' if ok > 0.5 else 'not supported'})
if {'sparse', 'soft', 'dense'} <= set(H5.index):
    m = H5['mean_over_contexts']; rows.append({'hyp': 'H5', 'variant': 'sparse', 'statistic': f"H(q): sparse {m['sparse']:.3f} < soft {m['soft']:.3f} ~ dense {m['dense']:.3f}", 'verdict': 'supported' if m['sparse'] < min(m['soft'], m['dense']) - 0.005 else 'not supported'})
summary = pd.DataFrame(rows); display(summary)
summary.to_csv(f'{OUT}/hypotheses_B.csv', index=False)
res_B.drop(columns=[c for c in res_B.columns if c.endswith('_pt')]).to_csv(f'{OUT}/results_B.csv', index=False)
H2.to_csv(f'{OUT}/H2.csv', index=False); H3.to_csv(f'{OUT}/H3.csv', index=False); H4.to_csv(f'{OUT}/H4.csv', index=False); H5.to_csv(f'{OUT}/H5.csv')
json.dump({'recipe': RECIPE, 'sparse_extra': SPARSE_EXTRA, 'seeds': SEEDS, 'n_folds': N_FOLDS}, open(f'{OUT}/summary_B.json', 'w'))
for f in glob.glob(f'{RUNS}/*/best.pt'): os.remove(f)

### Reading the outcomes (idea.md 9)
| Result | Meaning |
|---|---|
| B ~ A, S ~ 0 | router collapsed or nothing to exploit |
| B ~ A, S high, H3 fails | router learned the label; no gain |
| B > A, H3 fails | gain from extra per-sample capacity, not context (compare with FiLM) |
| B > A, H3 and H4 pass | claim supported **for reweighting**; selection vs. conditioning is settled only by B-sparse vs. FiLM |